<center><img src="@@KEEP_00000@@ />    
## <center>[mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
# <center> البرنامج التعليمي. Mlxtend.SFS: طريقة سهلة لاختيار الميزات
### <center> المؤلف: أنطون جيلمانوف، @wicker



# 1. مقدمة



"تعد هندسة الميزات واختيار الميزات من أهم عناصر تحليل البيانات والتعلم الآلي على الإطلاق". <br> 
<br> 
يمكنك قراءة هذه العبارة في العديد من المقالات أو الكتب، وهذه هي الحقيقة. ولكن لماذا نحتاج إلى تحديد الميزات؟
<br> 
<br> 
### 1. ميزات "صاخبة".
مهما كانت البيانات الجيدة التي لديك، هناك دائمًا بعض الميزات المفيدة، التي تساعدك على حل المشكلة وبعض الميزات المزعجة - غير المفيدة في نموذج التنبؤ الخاص بك. مثل هذه الميزات خطيرة، لأنها يمكن أن تؤدي إلى الإفراط في التجهيز. على العكس من ذلك، يمكن تحسين جودة النموذج الخاص بك على البيانات المعلقة عن طريق حذفها من مجموعة البيانات.



### 2. مشكلة الحساب



إذا كانت مجموعة البيانات تحتوي على مئات أو آلاف الميزات، أو المقدر المناسب، أو التحقق من الصحة المتبادل أو غير ذلك
يمكن أن يستغرق الحساب الكثير من الوقت. بالطبع، يمكننا استخدام PCA لتقليل أبعاد البيانات، ولكن في بعض الأحيان لا يكون ذلك متاحًا لمهمة العمل الحالية. من المستحيل شرح كيف يمكن للشركات تغيير "ميزة PCA الجديدة" للوصول إلى أهدافها - وهذا ما يسمى "مشكلة التفسير". لذا فإن اختيار الميزة مفيد في هذه الحالة.



### 3. هندسة الميزات


حسنًا، لدينا مجموعة بيانات جيدة. لكننا أضفنا العديد من الميزات المخصصة ~~ للتغلب على خط Kaggle الأساسي ~~ ونحن الآن نبحث عن كيفية تحديد الميزات التي تعمل على تحسين مقياس الجودة وأيها لا. حسنًا... يمكننا استخدام تنظيم L1، وسيحرك بعض الأوزان نحو 0. ولكن ماذا لو تمكنا من ملاءمة المقدر مع مجموعات فرعية مختلفة من الميزات الجديدة، وإضافة واحدة إذا كان مقياس الجودة يتزايد أو إزالة واحد إذا كان مقياس الجودة يتناقص، وتأكد من أن الحل لا يستغرق سوى سطرين من التعليمات البرمجية؟ <br><br> **Mlxtend SequentialFeatureSelector هو ما نحتاج إليه! **



<p style="text-align: center;">**في كل مرة نحاول اختيار أفضل الميزات :)**</p>



![صورة على الويب](https://i.giphy.com/media/5yLgoceFO3BdJW1zvFu/giphy.webp)



حسنًا، المقدمة غير الرسمية تقترب من نهايتها. حان الوقت لفهم بعض النظريات الرسمية.



# 2. تقديم إلى SequentialFeatureSelector



من فضلك، قم بتثبيت بعض المكتبات، إذا لم تكن موجودة في نظامك


In [ ]:
#!pip install pandas
#!pip install mlxtend
#!pip install scikit-learn
#!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from mlxtend.feature_selection import SequentialFeatureSelector
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler


Mlxtend SequentialFeatureSelector هي خوارزمية بحث جشعة تُستخدم لتقليل مساحة ميزة الأبعاد d الأولية إلى مساحة فرعية لميزة الأبعاد k حيث k < d.



هناك 4 نكهات مختلفة من SFAs متاحة عبر *SequentialFeatureSelector*:
* التحديد المتسلسل للأمام (SFS)
* التحديد المتسلسل للخلف (SBS)
* التحديد المتسلسل للأمام (SFFS)
* التحديد العائم المتسلسل للخلف (SBFS)
في الخوارزمية "الأمامية"، نبدأ بدون أي ميزات في مجموعتنا الفرعية ونضيف ميزة واحدة في كل تكرار، مما يؤدي إلى زيادة مقياس الجودة إلى الحد الأقصى. على العكس من ذلك، تبدأ الخوارزمية "الرجعية" بمجموعة فرعية كاملة من الميزات وتزيل ميزة واحدة في كل تكرار مما يؤدي إلى تعظيم جودة نموذجنا.يمكن اعتبار المتغيرات العائمة، SFFS وSBFS، بمثابة امتدادات لخوارزميات SFS وSBS الأبسط. تحتوي الخوارزميات العائمة على خطوة استبعاد أو تضمين إضافية لإزالة الميزات بمجرد تضمينها (أو استبعادها)، بحيث يمكن أخذ عينات لعدد أكبر من مجموعات مجموعة الميزات الفرعية.
دعونا نلقي نظرة على كل واحد منهم.



## التحديد التسلسلي للأمام (SFS)
### الإدخال: 
Y={y<sub>1</sub>,y<sub>2</sub>,...,y<sub>d</sub>}
* تأخذ خوارزمية SFS مجموعة الميزات ذات الأبعاد الكاملة كمدخلات.
### الإخراج: 
X<sub>k</sub>={x<sub>j</sub>|j=1,2,...,k;x<sub>j</sub>∈Y}، حيث k=(0,1,2,...,d)
* يُرجع SFS مجموعة فرعية من الميزات؛ عدد المعالم المحددة k، حيث k<d, has to be specified a priori.

### Initialization: 

X<sub>0</sub>=∅، k=0
* نقوم بتهيئة الخوارزمية بمجموعة فارغة ∅ ("مجموعة فارغة") بحيث يكون k=0 (حيث k هو حجم المجموعة الفرعية).
### الخطوة 1 (التضمين):
x<sup>+</sup> = arg max J(x<sub>k</sub>+x)، حيث x∈Y−X<sub>k</sub>
X<sub>k+1</sub>=X<sub>k</sub>+x<sup>+</sup>
ك=ك+1 
#### انتقل إلى الخطوة 1
* في هذه الخطوة، نضيف ميزة إضافية، x<sup>+</sup>، إلى مجموعة الميزات الفرعية X<sub>k</sub>.
* x<sup>+</sup> هي الميزة التي تزيد من وظيفة المعيار لدينا، أي الميزة المرتبطة بأفضل أداء للمصنف إذا تمت إضافتها إلى X<sub>k</sub>.
* نكرر هذا الإجراء حتى يتم استيفاء معيار الإنهاء.
### الإنهاء: 
ك=صنضيف ميزات من مجموعة الميزات الفرعية Xk حتى تحتوي مجموعة الميزات الفرعية بالحجم k على عدد الميزات المطلوبة p التي حددناها مسبقًا.



___



## التحديد التسلسلي للخلف (SBS)
### الإدخال: 
مجموعة جميع الميزات: Y={y<sub>1</sub>,y<sub>2</sub>,...,y<sub>d</sub>}
* تأخذ خوارزمية SBS مجموعة الميزات بأكملها كمدخلات.
### الإخراج: 
X<sub>k</sub>={x<sub>j</sub>|j=1,2,...,k;x<sub>j</sub>∈Y}، حيث k=(0,1,2,...,d)
* تقوم SBS بإرجاع مجموعة فرعية من الميزات؛ عدد المعالم المحددة k، حيث k<d, has to be specified a priori.

### Initialization: 

X<sub>0</sub>=Y، k=d
* نقوم بتهيئة الخوارزمية باستخدام مجموعة الميزات المحددة بحيث يكون k=d.
### الخطوة 1 (الاستبعاد):
x<sup>-</sup>= arg max J(x<sub>k</sub>-x)، حيث x∈X<sub>k </sub>
X<sub>k-1</sub>=X<sub>k</sub>-x<sup>-</sup>
ك=ك-1 
#### انتقل إلى الخطوة 1
* في هذه الخطوة، نقوم بإزالة الميزة، x<sup>-</sup> من مجموعة الميزات الفرعية X<sub>k</sub>.
* x<sup>-</sup> هي الميزة التي تعمل على زيادة وظيفة المعيار لدينا عند الإزالة، أي الميزة المرتبطة بأفضل أداء للمصنف إذا تمت إزالته من X<sub>k</sub>.
* نكرر هذا الإجراء حتى يتم استيفاء معيار الإنهاء.
### الإنهاء: 
ك=ص
نقوم بإزالة الميزات من مجموعة الميزات الفرعية X<sub>k</sub> حتى تحتوي مجموعة الميزات الفرعية بالحجم k على عدد الميزات المطلوبة p التي حددناها مسبقًا



***



## التحديد المتسلسل للأمام (SFFS)
### الإدخال:مجموعة جميع الميزات: Y={y<sub>1</sub>,y<sub>2</sub>,...,y<sub>d</sub>}
* تأخذ خوارزمية SFFS مجموعة الميزات بأكملها كمدخلات
### الإخراج: 
X<sub>k</sub>={x<sub>j</sub>|j=1,2,...,k;x<sub>j</sub>∈Y}، حيث k=(0,1,2,...,d)
* الناتج الذي تم إرجاعه من الخوارزمية هو مجموعة فرعية من مساحة الميزة ذات الحجم المحدد.
### التهيئة: 
X<sub>0</sub>=∅، ك=0
* نقوم بتهيئة الخوارزمية بمجموعة فارغة ("مجموعة فارغة") بحيث يكون k = 0 (حيث k هو حجم المجموعة الفرعية) 
### الخطوة 1 (التضمين):
x<sup>+</sup>= arg max J(x<sub>k</sub>+x)، حيث x∈Y−X<sub>k </sub>
X<sub>k+1</sub>=X<sub>k</sub>+x<sup>+</sup>
ك=ك+1 
#### انتقل إلى الخطوة 2
في الخطوة 1، نقوم بتضمين الميزة من مساحة الميزة التي تؤدي إلى أفضل زيادة في الأداء لمجموعة الميزات الفرعية لدينا (يتم تقييمها بواسطة وظيفة المعيار). ثم ننتقل إلى الخطوة 2.
### الخطوة الثانية (الاستبعاد المشروط):
x<sup>-</sup>= arg max J(x<sub>k</sub>-x)، حيث x∈X<sub>k</sub>
إذا كان J(x<sub>k</sub> - x) > J(x<sub>k</sub>):
   X<sub>k-1</sub>=X<sub>k</sub>-x<sup>-</sup> 
   ك=ك-1 
     
#### انتقل إلى الخطوة 1
في الخطوة 2، نقوم بإزالة الميزة فقط إذا كانت المجموعة الفرعية الناتجة ستحصل على زيادة في الأداء. إذا كان k=2 أو لا يمكن إجراء تحسين (أي، لا يمكن العثور على هذه الميزة x<sup>-</sup>)، فارجع إلى الخطوة 1؛ وإلا، كرر هذه الخطوة.
يتم تكرار الخطوتين 1 و2 حتى يتم الوصول إلى معيار الإنهاء.
### الإنهاء: 
ك=صنضيف ميزات من مجموعة الميزات الفرعية X<sub>k</sub> حتى تحتوي مجموعة الميزات الفرعية بالحجم k على عدد الميزات المطلوبة p التي حددناها مسبقًا.



***



## التحديد العائم المتسلسل للخلف (SBFS)
### الإدخال: 
مجموعة جميع الميزات: Y={y<sub>1</sub>,y<sub>2</sub>,...,y<sub>d</sub>}
* تأخذ خوارزمية SBFS مجموعة الميزات بأكملها كمدخلات.
### الإخراج: 
X<sub>k</sub>={x<sub>j</sub>|j=1,2,...,k;x<sub>j</sub>∈Y}، حيث k=(0,1,2,...,d)
* تقوم SBFS بإرجاع مجموعة فرعية من الميزات؛ عدد الميزات المحددة k، حيث k<d, has to be specified a priori.

### Initialization: 

X<sub>0</sub>=Y، k=d
* نقوم بتهيئة الخوارزمية باستخدام مجموعة الميزات المحددة بحيث يكون k=d.
### الخطوة 1 (الاستبعاد):
x<sup>-</sup>= arg max J(x<sub>k</sub>-x)، حيث x∈X<sub>k </sub>
X<sub>k-1</sub>=X<sub>k</sub>-x<sup>-</sup>
ك=ك-1 
#### انتقل إلى الخطوة 2
* في هذه الخطوة، نقوم بإزالة الميزة، x<sup>-</sup> من مجموعة الميزات الفرعية X<sub>k</sub>.
* x<sup>-</sup> هي الميزة التي تعمل على زيادة وظيفة المعيار لدينا عند الإزالة، أي الميزة المرتبطة بأفضل أداء للمصنف إذا تمت إزالته من X<sub>k</sub>.
### الخطوة الثانية (الإدراج المشروط):
x<sup>+</sup>= arg max J(x<sub>k</sub>+x)، حيث x∈Y−X<sub>k</sub>
إذا كان J(x<sub>k</sub> + x<sup>+</sup>) > J(x<sub>k</sub>):
   X<sub>k+1</sub>=X<sub>k</sub>+x<sup>+</sup>   ك=ك+1 
     
#### انتقل إلى الخطوة 1
في الخطوة 2، نبحث عن الميزات التي تعمل على تحسين أداء المصنف إذا تمت إضافتها مرة أخرى إلى مجموعة الميزات الفرعية. في حالة وجود مثل هذه الميزات، فإننا نضيف الميزة x<sup>+</sup> والتي يتم من خلالها زيادة تحسين الأداء إلى الحد الأقصى. إذا تعذر إجراء k=2 أو تحسين (أي، لا يمكن العثور على هذه الميزة x<sup>+</sup>)، فارجع إلى الخطوة 1؛ وإلا، كرر هذه الخطوة.
### الإنهاء: 
ك=ص
نضيف ميزات من مجموعة الميزات الفرعية X<sub>k</sub> حتى تحتوي مجموعة الميزات الفرعية بالحجم k على عدد الميزات المطلوبة p التي حددناها مسبقًا.



# 3. كائن SequentialFeatureSelector



دعونا نلقي نظرة على وثائق ومعلمات كائن SFS



**SequentialFeatureSelector**(المقدر، k_features=1، الأمام=صحيح، العائم=خطأ، مطول=0، التسجيل=لا شيء، السيرة الذاتية=5، n_jobs=1، pre_dispatch='2n_jobs'، clone_estimator=True)
اختيار الميزة التسلسلية للتصنيف والانحدار.
_**المعلمات**_
**المُقدِّر**: المُصنف أو المُتراجع scikit-Learn
**k_features** : int أو tuple أو str (الافتراضي: 1)عدد الميزات المراد تحديدها، حيث k_features < مجموعة الميزات الكاملة. يمكن توفير صف يحتوي على قيمة الحد الأدنى والحد الأقصى، وسيأخذ SFS في الاعتبار إرجاع أي مجموعة ميزات بين الحد الأدنى والحد الأقصى التي سجلت أعلى مستوى في التحقق المتبادل. على سبيل المثال، سوف يقوم الصف (1، 4) بإرجاع أي مجموعة من 1 إلى 4 ميزات بدلاً من عدد ثابت من الميزات k. وسيطة سلسلة "الأفضل" أو "البخيل". إذا تم توفير "الأفضل"، فسيقوم محدد الميزة بإرجاع مجموعة الميزات الفرعية بأفضل أداء للتحقق المتبادل. إذا تم توفير "بخل" كوسيطة، فسيتم تحديد أصغر مجموعة فرعية من الميزات التي تقع ضمن خطأ قياسي واحد في أداء التحقق المتبادل.
** الأمام ** : منطقي (الافتراضي: صحيح)
التحديد للأمام إذا كان صحيحًا، والاختيار للخلف بخلاف ذلك
**العائمة**: منطقي (الافتراضي: خطأ)
يضيف استبعاد/تضمين مشروط إذا كان صحيحًا.
**verbose** : int (افتراضي: 0)، مستوى الإسهاب المستخدم في التسجيل.
إذا كانت 0، فلا يوجد إخراج، وإذا كان هناك عدد واحد من الميزات في المجموعة الحالية، وإذا كان هناك تسجيلان تفصيليان، بما في ذلك الطابع الزمني ودرجات السيرة الذاتية في الخطوة.
**تسجيل النقاط**: سلسلة أو قابلة للاستدعاء أو لا شيء (الافتراضي: لا شيء)
إذا لم يكن هناك شيء (افتراضي)، فسيتم استخدام "الدقة" لمصنفات sklearn و"r2" لمصنفات تراجعات sklearn. إذا كان str، يستخدم معرف سلسلة قياس sklearn، على سبيل المثال {accuracy, f1, Precision, Recall, roc_auc} للمصنفات، {'mean_absolute_error', 'mean_squared_error'/'neg_mean_squared_error', 'median_absolute_error', 'r2'} للتراجعات.
** السيرة الذاتية ** : int (الافتراضي: 5)
قطار إنتاج صحيح أو قابل للتكرار، انقسامات الاختبار. إذا كان cv عددًا صحيحًا وكان المقدر عبارة عن مصنف (أو يتكون y من تسميات فئة عددية صحيحة)، فسيتم تقسيمه إلى k-fold. وبخلاف ذلك، يتم إجراء التحقق المتبادل المنتظم على شكل k-fold. لا يوجد تحقق متقاطع إذا كانت السيرة الذاتية هي لا شيء أو خطأ أو 0.
**n_jobs** : int (الافتراضي: 1)عدد وحدات المعالجة المركزية التي سيتم استخدامها لتقييم مجموعات فرعية مختلفة من الميزات بالتوازي. -1 يعني "جميع وحدات المعالجة المركزية".
**pre_dispatch** : int، أو سلسلة (الافتراضي: '2*n_jobs')
يتحكم في عدد المهام التي يتم إرسالها أثناء التنفيذ المتوازي إذا كانت n_jobs > 1 أو n_jobs=-1. يمكن أن يكون تقليل هذا الرقم مفيدًا لتجنب زيادة استهلاك الذاكرة عند إرسال مهام أكثر مما تستطيع وحدات المعالجة المركزية (CPUs) معالجته. يمكن أن تكون هذه المعلمة: لا شيء، وفي هذه الحالة يتم إنشاء كافة الوظائف وإنشاءها على الفور. استخدم هذا للمهام خفيفة الوزن وسريعة التشغيل، لتجنب التأخير بسبب نشر الوظائف عند الطلب، وهو int، معطيًا العدد الدقيق لإجمالي المهام التي تم إنشاؤها سلسلة، مما يعطي تعبيرًا كدالة لـ n_jobs، كما في 2*n_jobs
**clone_estimator**: منطقي (الافتراضي: صحيح)
مقدر النسخ إذا كان صحيحا؛ يعمل مع مثيل المقدر الأصلي إذا كان False. اضبط على False إذا لم يقم المُقدِّر بتطبيق أساليب set_params وget_params الخاصة بـ scikit-learn. بالإضافة إلى ذلك، يلزم تعيين cv=0 وn_jobs=1.



# 4. الانحدار اللوجستي مع اختيار الميزة بواسطة mlxtend.sfs



في هذه المقالة سوف نستخدم مجموعة بيانات Toy sklearn **"breast_cancer"** (مهمة تصنيف ثنائية). يتيح تحميل مجموعة البيانات وإلقاء نظرة على البيانات.


In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

In [ ]:
df, y = pd.DataFrame(data=data.data, columns=data.feature_names), data.target

In [ ]:
df.info()

In [ ]:
columns = df.columns

In [ ]:
df.head()

In [ ]:
sum(y) / len(y)


هناك 30 ميزة عائمة غير فارغة و569 مثالًا. 62,7% من الأمثلة تحتوي على الفئة 1 و37,3% من الأمثلة تحتوي على الفئة 0. الفئات ليست منحرفة جدًا، لذا فإن مقياس الدقة مناسب لنا.



سوف نستخدم LogisticRegression كخوارزمية أساسية لدينا. أولاً، تحجيم البيانات. 


In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)


ثم بدء كائن التحقق المتقاطع بخمسة طيات مع حالة عشوائية ثابتة 


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=17)


التحقق من نتائج السيرة الذاتية دون ضبط المعلمات


In [ ]:
logit = LogisticRegression()
cross_val_score(logit, df_scaled, y, cv=cv).mean()


نتيجة السيرة الذاتية **0.984**
سيكون خط الأساس لدينا. تحاول التغلب عليه مع اختيار الميزةسنستخدم التحديد التسلسلي للخلف، لذا قم بتبديل المعلمات **forward** و**floating** إلى **False**
التحديد التسلسلي للخلف يعني أن:
* سنبدأ بجميع الميزات K (في مجموعة البيانات الخاصة بنا K=30)
* في كل تكرار، نلائم المقدر مع ميزات K-n ونحافظ على مجموعة K-n الفرعية من الميزات مع أفضل النتائج
تعيين المعلمة k_features على الصف (1، K)، لذلك ستكون مجموعة فرعية من الميزات في النطاق (1، 30) مع أفضل تسجيل في السيرة الذاتية كمخرجات لطريقة fit_transform.


In [ ]:
logit = LogisticRegression()
sbs = SequentialFeatureSelector(
    logit,
    k_features=(1, 30),
    forward=False,
    floating=False,
    verbose=2,
    scoring="accuracy",
    cv=cv,
)


توجد معلومات حول تسجيل السيرة الذاتية في كل تكرار في السجل. أفضل جودة لدينا مع مجموعة فرعية تحتوي على 15 ومن 17 إلى 24 ميزة.


In [ ]:
X_sbs = sbs.fit_transform(df_scaled, y, custom_feature_names=columns)


نتائج التخطيط:


In [ ]:
plot_sfs(sbs.get_metric_dict(), kind="std_dev");


تقوم SBS بإرجاع مجموعة فرعية من إطار البيانات مع ميزات K المثالية


In [ ]:
X_sbs.shape


هناك مجموعة فرعية من أسماء الميزات المحددة:


In [ ]:
sbs.k_feature_names_

In [ ]:
"The best quality is {} with {} features in dataset".format(
    sbs.k_score_, len(sbs.k_feature_idx_)
)


يتم زيادة الجودة! ***0.984 -> 0.988***



حفظ النتائج لإملاء وتجربة خوارزميات SFS أخرى


In [ ]:
sbs_dict = dict()
for i in sbs.subsets_.values():
    sbs_dict[len(i["feature_names"])] = i["avg_score"]


***



نحاول الآن استخدام التحديد التسلسلي للأمام، لذا قم بتبديل المعلمة للأمام إلى **True** <br><br>
التحديد التسلسلي للأمام يعني أن:
* سنبدأ ** بـ 0 ميزات **
* في كل تكرار N، نلائم المقدر مع ميزات N ونحافظ على مجموعة N فرعية من الميزات ذات أفضل النتائج


In [ ]:
logit = LogisticRegression()
sfs = SequentialFeatureSelector(
    logit,
    k_features=(1, 30),
    forward=True,
    floating=False,
    verbose=2,
    scoring="accuracy",
    cv=cv,
)

In [ ]:
X_sfs = sfs.fit_transform(df_scaled, y)


نتائج التوطين:


In [ ]:
plot_sfs(sfs.get_metric_dict(), kind="std_dev");

In [ ]:
"The best quality is {} with {} features in dataset".format(
    sfs.k_score_, len(sfs.k_feature_idx_)
)


الآن الجودة تساوي خط الأساس لدينا. 


لماذا جودة SFS أسوأ من SBS؟ 
نحن نستخدم خوارزمية "الأمام"، لذا في التكرار الأول نختار ميزة واحدة ونلائم المُقدِّر معها. من الواضح أن العثور على الميزة "الأفضل" التي تناسب مجموعة بيانات أحادية البعد ليس فعالاً للغاية. أكثر من ذلك، في التحديد التسلسلي للأمام، لا يمكننا إزالة الميزة بمجرد إضافتها. <br><br> دعنا نحاول العثور على "الميزة السيئة" التي نضيفها إلى مجموعة البيانات الخاصة بنا مرة واحدة وفي أي مرحلة تكرار حدث هذا.


In [ ]:
sbs_feat = set(sbs.subsets_[24]["feature_idx"])  # best feature set of SBS algorithm
for i in range(1, 30):
    sfs_feat = set(
        sfs.subsets_[i]["feature_idx"]
    )  # iterate throw feature set on each iteration of SFS algorithm
    if len([x for x in sfs_feat if x not in sbs_feat]) > 0:
        print(
            'We add "bad feature" # {} on {} iteration stage'.format(
                sfs_feat - sbs_feat, i
            )
        )
        break


احفظ النتائج في كل تكرار أيضًا


In [ ]:
sfs_dict = dict()
for i in sfs.subsets_.values():
    sfs_dict[len(i["feature_names"])] = i["avg_score"]


سنحاول الآن استخدام التحديد المتسلسل للأمام العائم، لذا قم بتبديل المعلمة العائمة إلى True. يمكن أن يساعدنا في إزالة أسوأ ميزة في كل خطوة إضافية للتكرار


In [ ]:
logit = LogisticRegression()
sffs = SequentialFeatureSelector(
    logit,
    k_features=(1, 30),
    forward=True,
    floating=True,
    verbose=2,
    scoring="accuracy",
    cv=cv,
)

In [ ]:
X_sffs = sffs.fit_transform(df_scaled, y)


نتائج التخطيط:


In [ ]:
plot_sfs(sffs.get_metric_dict(), kind="std_dev");

In [ ]:
"The best quality is {} with {} features in dataset".format(
    sffs.k_score_, len(sffs.k_feature_idx_)
)


الجودة أعلى قليلاً من SFS، لكن SBS هي أفضل خوارزمية اليوم. حفظ النتائج للإملاء ودعنا نجرب التنفيذ الأخير - التحديد المتسلسل العائم للخلف


In [ ]:
sffs_dict = dict()
for i in sffs.subsets_.values():
    sffs_dict[len(i["feature_names"])] = i["avg_score"]

In [ ]:
logit = LogisticRegression()
sbfs = SequentialFeatureSelector(
    logit,
    k_features=(1, 30),
    forward=False,
    floating=True,
    verbose=2,
    scoring="accuracy",
    cv=cv,
)

In [ ]:
X_sbfs = sbfs.fit_transform(df_scaled, y)


نتائج التخطيط:


In [ ]:
plot_sfs(sbfs.get_metric_dict(), kind="std_dev");

In [ ]:
"The best quality is {} with {} features in dataset".format(
    sbfs.k_score_, len(sbfs.k_feature_idx_)
)


جودة خوارزميات SBS وSBFS متساوية في مثالنا. لكن في بعض الأحيان زادت.


In [ ]:
sbfs_dict = dict()
for i in sbfs.subsets_.values():
    sbfs_dict[len(i["feature_names"])] = i["avg_score"]


# 5. مقارنة النتائج مع RFE وPCA



تجربة اختيار ميزة أخرى وخوارزميات تقليل الأبعاد


In [ ]:
dict_pca = dict()
for i in range(1, 31):
    pca = PCA(n_components=i)
    df_pca = pca.fit_transform(df_scaled, y)
    logit = LogisticRegression()
    score = cross_val_score(logit, df_pca, y, cv=cv).mean()
    dict_pca[i] = score

In [ ]:
"The best quality is {} with {} features in dataset".format(
    max(dict_pca.values()), max(dict_pca, key=dict_pca.get)
)


مقياس الدقة أقل في مجموعة بيانات PCA


In [ ]:
dict_rfe = dict()
for i in range(1, 31):
    rfe = RFE(logit, n_features_to_select=i)
    df_rfe = rfe.fit_transform(df_scaled, y)
    logit = LogisticRegression()
    score = cross_val_score(logit, df_rfe, y, cv=cv).mean()
    dict_rfe[i] = score

In [ ]:
"The best quality is {} with {} features in dataset".format(
    max(dict_rfe.values()), max(dict_rfe, key=dict_rfe.get)
)


جودة RFE أقل أيضًا. يعد RFE أقل تعقيدًا من الناحية الحسابية باستخدام معاملات وزن الميزة (على سبيل المثال، النماذج الخطية) أو أهمية الميزة (الخوارزميات القائمة على الشجرة) لإزالة الميزات بشكل متكرر، في حين تقوم SFSs بإزالة (أو إضافة) الميزات بناءً على مصنف محدد من قبل المستخدم/مقياس أداء الانحدار.



مقارنة درجات السيرة الذاتية لجميع الخوارزميات


In [ ]:
pd.DataFrame(
    data=[pd.Series(dict_rfe), pd.Series(dict_pca), pd.Series(sbs_dict)],
    index=["RFE", "PCA", "SBS"],
).T

الحد الأقصى من الدرجات التي حصلنا عليها مع SBS والحد الأدنى 15 ميزة في المجموعة الفرعية. RFE أسوأ مع أي عدد من الميزات. PCA أفضل فقط مع 9 ميزات في المجموعة الفرعية. لم يتمكن RFE وPCA من العثور على مجموعة فرعية من الميزات ذات نقاط أكثر من نقاط مجموعة البيانات الكاملة. SBS ديل معها.



<p style="text-align: center;">**كيف سنختار الميزات بعد هذا البرنامج التعليمي :)**</p>



<img src="@@KEEP_00003@@ />



 وبطبيعة الحال، فإن RFE و PCA و SBS يحلون مهام مختلفة قليلاً. ومن المهم أن نعرف كيف ومتى يتعين علينا تنفيذ هذا الصك أو ذاك. والأهم من ذلك هو أن يكون لديك عقل مستفسر 
واختبار الفرضيات الأكثر جنونا :)



#الخلاصة 



في هذا البرنامج التعليمي، درسنا شيئًا جديدًا حول اختيار الميزات، وفهمنا كيفية عمل SequentialFeatureSelector من مكتبة Mlxtend - فهو يسمح بالاختيار السهل جدًا من الميزات الجديدة التي تم إنشاؤها وتعزيز جودة النموذج. ثم قمنا بمقارنتها مع اختيار ميزة أخرى وخوارزميات تقليل الأبعاد. <br><br>
غالبًا ما يولي علماء البيانات المبتدئون القليل من الاهتمام لاختيار الميزات ويحاولون اختبار العديد من النماذج المختلفة بدلاً من ذلك. لكن اختيار الميزة يمكن أن يعزز نتيجة النموذج كثيرًا. يكاد يكون من المستحيل الحصول على أفضل Kaggle دون ~~تكديس xgboost~~ هندسة الميزات الدقيقة واختيار أفضل الميزات.
احفظ الأفضل واحذف الباقي! هذا كل شيء يا رفاق!



### روابط مميزة
مستندات Mlxtend الرسمية https://rasbt.github.io/mlxtend/user_guide/feature_selection/SequentialFeatureSelector